# NSGA-II vs NSGA-II Partitioned (6 niches) — Kaggle Notebook

This notebook benchmarks **NSGA2** vs **NSGA2-PARTITIONED** from the repo. It:

1. Clones the repository
2. Downloads benchmark files from Google Drive (via `gdown`)
3. Runs multiple independent seeds for both algorithms
4. Extracts the final Pareto archives
5. Computes **Hypervolume (HV)** and **IGD** against a reference (‘theoretical’) Pareto front
6. Plots both Pareto fronts + the reference front

Notes:
- The niche partitioning is defined for **NAS-Bench-201** only.
- If NAS-Bench-201 accuracy is unavailable, the notebook still runs in hardware-only mode.

In [ ]:
# ===== USER SETTINGS (EDIT THESE) =====

# Git repo URL to clone (must contain HW-NAS-Bench-MetaHeuristicsOptim code).
REPO_URL = "https://github.com/<USER>/<REPO>.git"  # TODO: replace
REPO_DIRNAME = "HW-NAS-Bench-MetaHeuristicsOptim"  # folder inside repo (usually this)

# Google Drive IDs or share links
HW_PICKLE_ID_OR_LINK = "<HW_PICKLE_FILE_ID>"  # HW-NAS-Bench-v1_0.pickle
NB201_PTH_ID_OR_LINK = "<NB201_PTH_FILE_ID>"      # NAS-Bench-201-v1_1-096897.pth (optional)

# Experiment parameters
SEARCH_SPACE = "nasbench201"  # required
DEVICE = "edgegpu"
DATASET = "cifar10"

NUM_RUNS = 5
POPULATION_TOTAL = 60   # total pop budget; partitioned splits across 6 niches
ITERATIONS = 50
SEED_BASE = 42

# Objective pair for metrics/plots (minimization space):
# - ('latency', 'energy') is fully supported from HW pickle only
# - ('latency', 'error') requires accuracy to be meaningful (needs NAS-Bench-201 .pth + nas-bench-201 pkg)
OBJECTIVE_PAIR = ("latency", "energy")

# Reference front computation:
# For ('latency','energy'): FULL is fast (15625 points).
# If you include 'error', full evaluation requires computing accuracy for many architectures and may be slow.
THEORETICAL_EVAL_LIMIT = None  # None = full space; or set e.g. 3000 for faster reference

# =====================================

In [ ]:
# Install runtime dependencies (Kaggle has many already; these are lightweight).
!pip -q install gdown pymoo seaborn tqdm

# Optional: enables accuracy/error objective if you also downloaded the NAS-Bench-201 .pth.
# This may fail on some Kaggle images depending on Python version; we continue in hardware-only mode.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "nas-bench-201"], check=False)

In [ ]:
import os
import re
import sys
import json
import math
import time
import pickle
from dataclasses import asdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from pymoo.indicators.hv import HV
from pymoo.indicators.igd import IGD

sns.set_context('talk')
plt.rcParams['figure.figsize'] = (10, 7)

WORKDIR = Path('/kaggle/working')
WORKDIR

In [ ]:
def extract_gdrive_file_id(id_or_link: str) -> str:
    s = (id_or_link or '').strip()
    if not s:
        return ''
    # If already looks like an ID (alnum, -,_), keep it
    if re.fullmatch(r'[A-Za-z0-9_-]{10,}', s):
        return s
    # Common link patterns
    m = re.search(r'/d/([A-Za-z0-9_-]+)', s)
    if m:
        return m.group(1)
    m = re.search(r'[?&]id=([A-Za-z0-9_-]+)', s)
    if m:
        return m.group(1)
    raise ValueError(f'Could not extract Google Drive file ID from: {id_or_link!r}')

def gdrive_download(file_id_or_link: str, out_path: Path) -> Path:
    import gdown
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists() and out_path.stat().st_size > 0:
        return out_path
    fid = extract_gdrive_file_id(file_id_or_link)
    url = f'https://drive.google.com/uc?id={fid}'
    print('Downloading:', out_path.name)
    gdown.download(url, str(out_path), quiet=False)
    if not out_path.exists() or out_path.stat().st_size == 0:
        raise RuntimeError(f'Download failed or empty file: {out_path}')
    return out_path

def ensure_repo_cloned(repo_url: str, workdir: Path = WORKDIR) -> Path:
    repo_url = repo_url.strip()
    if not repo_url or '<USER>' in repo_url or '<REPO>' in repo_url:
        raise ValueError('Please set REPO_URL to a real Git URL.')
    name = repo_url.rstrip('/').split('/')[-1]
    if name.endswith('.git'):
        name = name[:-4]
    target = workdir / name
    if target.exists():
        print('Repo already exists:', target)
        return target
    import subprocess
    subprocess.run(['git', 'clone', repo_url, str(target)], check=True)
    return target

In [ ]:
repo_root = ensure_repo_cloned(REPO_URL)
project_root = repo_root / REPO_DIRNAME
if not project_root.exists():
    raise FileNotFoundError(f'Expected project folder not found: {project_root}')

# Add project to import path (avoid packaging; pyproject requires Python>=3.13).
sys.path.insert(0, str(project_root))
print('Project root:', project_root)

In [ ]:
# Download benchmark files into repo's data/ folder
data_dir = project_root / 'data'
pickle_path = data_dir / 'HW-NAS-Bench-v1_0.pickle'
nb201_path = data_dir / 'NAS-Bench-201-v1_1-096897.pth'

gdrive_download(HW_PICKLE_ID_OR_LINK, pickle_path)

# NB201 is optional (for accuracy). If you don't want it, set NB201_PTH_ID_OR_LINK to ''
if NB201_PTH_ID_OR_LINK and '<NB201' not in NB201_PTH_ID_OR_LINK:
    try:
        gdrive_download(NB201_PTH_ID_OR_LINK, nb201_path)
    except Exception as e:
        print('NB201 .pth download failed (continuing hardware-only):', e)
else:
    print('Skipping NAS-Bench-201 .pth download (hardware-only).')

print('Files:', pickle_path.exists(), nb201_path.exists())
print('HW pickle size (MB):', round(pickle_path.stat().st_size/1e6, 1))
if nb201_path.exists():
    print('NB201 pth size (MB):', round(nb201_path.stat().st_size/1e6, 1))

In [ ]:
from core.api import HWNASBenchAPI
from core.fitness import HardwareAwareFitness, NASBench201AccuracyAPI
from core.types import VALID_DATASETS, VALID_DEVICES
from algorithms.metaheuristics import get_algorithm
from algorithms.multiobjective.pareto import fast_non_dominated_sort

assert SEARCH_SPACE == 'nasbench201', 'This notebook supports partitioned niches only for nasbench201.'
assert DEVICE in VALID_DEVICES, f'Invalid DEVICE. Valid: {VALID_DEVICES}'
assert DATASET in VALID_DATASETS, f'Invalid DATASET. Valid: {VALID_DATASETS}'

api = HWNASBenchAPI(pickle_path, search_space=SEARCH_SPACE)
api

In [ ]:
# Fitness object is used to compute normalized objectives consistent with the algorithms.
fitness = HardwareAwareFitness(
    api,
    nb201_path=str(nb201_path) if nb201_path.exists() else None,
    target_device=DEVICE,
    dataset=DATASET,
    latency_weight=0.5,
    energy_weight=0.2,
    accuracy_weight=0.3,
)
print(fitness)

In [ ]:
def select_objective_columns(objectives_3d: np.ndarray, pair: tuple[str, str]) -> np.ndarray:
    # The repo uses compute_multi() -> (lat_norm, eng_norm, err_norm), all minimization
    idx = {'latency': 0, 'energy': 1, 'error': 2}
    a, b = pair
    # Allow using 'accuracy' as an alias: we represent it as 'error' (minimization) = 1 - acc/acc_baseline
    if a == 'accuracy':
        a = 'error'
    if b == 'accuracy':
        b = 'error'
    if a not in idx or b not in idx:
        raise ValueError(f'OBJECTIVE_PAIR must use keys {list(idx)} (plus alias accuracy), got {pair}')
    return objectives_3d[:, [idx[a], idx[b]]]

def nondominated_front(points: np.ndarray) -> np.ndarray:
    # Returns subset of points belonging to first Pareto front (minimization).
    fronts = fast_non_dominated_sort(points)
    if not fronts:
        return points
    return points[fronts[0]]

def compute_reference_front_from_hw_pickle(
    pickle_path: Path,
    dataset: str,
    device: str,
    fitness_obj: HardwareAwareFitness,
    eval_limit: int | None = None,
    use_accuracy: bool = False,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Returns (ref_front_2d, all_points_2d, all_points_3d_norm).

    - all_points_3d_norm is (lat_norm, eng_norm, err_norm) for the evaluated set
    - all_points_2d is the selected objective-pair projection
    - ref_front_2d is the non-dominated set on all_points_2d
    """
    data = pickle.load(open(pickle_path, 'rb'))
    nb = data['nasbench201'][dataset]

    lat_key = f'{device}_latency'
    eng_key = f'{device}_energy'
    if lat_key not in nb:
        raise KeyError(f'Missing {lat_key} in pickle for {dataset}')

    lat = np.asarray(nb[lat_key], dtype=float)
    if eng_key in nb:
        eng = np.asarray(nb[eng_key], dtype=float)
    else:
        # Some devices might not have energy; use zeros (still allows latency-only tradeoffs).
        eng = np.zeros_like(lat)

    n = len(lat)
    idxs = np.arange(n)
    if eval_limit is not None and eval_limit < n:
        rng = np.random.default_rng(0)
        idxs = rng.choice(idxs, size=int(eval_limit), replace=False)
        idxs.sort()

    lat = lat[idxs]
    eng = eng[idxs]

    lat_norm = lat / fitness_obj.lat_baseline if fitness_obj.lat_baseline > 0 else lat
    eng_norm = eng / fitness_obj.eng_baseline if fitness_obj.eng_baseline > 0 else eng

    # Accuracy / error
    if use_accuracy:
        try:
            acc_api = NASBench201AccuracyAPI(str(nb201_path), hp='200') if nb201_path.exists() else None
            if acc_api is None or not acc_api.available:
                raise RuntimeError('NAS-Bench-201 API unavailable')
            acc = np.zeros(len(idxs), dtype=float)
            for j, arch_idx in enumerate(tqdm(idxs, desc='Reference accuracy')):
                acc[j] = acc_api.get_accuracy(int(arch_idx), dataset=DATASET)
        except Exception as e:
            print('Accuracy unavailable for reference front (using constant error=1).', e)
            acc = np.zeros(len(idxs), dtype=float)
    else:
        acc = np.zeros(len(idxs), dtype=float)

    acc_baseline = fitness_obj.acc_baseline if fitness_obj.acc_baseline > 0 else 100.0
    err_norm = 1.0 - (acc / acc_baseline)

    all_3d = np.vstack([lat_norm, eng_norm, err_norm]).T
    all_2d = select_objective_columns(all_3d, OBJECTIVE_PAIR)
    ref_2d = nondominated_front(all_2d)
    return ref_2d, all_2d, all_3d

use_acc_for_ref = any(o in ('error', 'accuracy') for o in OBJECTIVE_PAIR)
ref_front_2d, all_points_2d, all_points_3d = compute_reference_front_from_hw_pickle(
    pickle_path=pickle_path,
    dataset=DATASET,
    device=DEVICE,
    fitness_obj=fitness,
    eval_limit=THEORETICAL_EVAL_LIMIT,
    use_accuracy=use_acc_for_ref,
 )
print('Reference front size:', len(ref_front_2d), 'out of', len(all_points_2d))

In [ ]:
# Hypervolume reference point: must be worse than all points (minimization).
max_vals = np.max(all_points_2d, axis=0)
min_vals = np.min(all_points_2d, axis=0)
ref_point = max_vals + 0.1 * (max_vals - min_vals + 1e-12)
print('ref_point:', ref_point)

hv = HV(ref_point=ref_point)
igd = IGD(ref_front_2d)

In [ ]:
def run_one(algo_key: str, seed: int) -> dict:
    AlgoClass = get_algorithm(algo_key)

    # New fitness per run so query statistics are per-run (but the API is a fast lookup).
    fit = HardwareAwareFitness(
        api,
        nb201_path=str(nb201_path) if nb201_path.exists() else None,
        target_device=DEVICE,
        dataset=DATASET,
        latency_weight=0.5,
        energy_weight=0.2,
        accuracy_weight=0.3,
    )

    optimizer = AlgoClass(
        fitness_function=fit,
        search_space_size=api.search_space_size(DATASET),
        dim=api.search_dimension(DATASET),
        population_size=POPULATION_TOTAL,
        max_iterations=ITERATIONS,
        seed=seed,
    )

    t0 = time.perf_counter()
    _best_sol, best_fit, _hist = optimizer.optimize()
    elapsed = time.perf_counter() - t0

    # Extract Pareto archive from optimizer if available
    arch_obj = getattr(optimizer, 'archive_objectives', None)
    if arch_obj is None or len(arch_obj) == 0:
        raise RuntimeError(f'{algo_key} did not expose archive_objectives')

    front3 = np.asarray(arch_obj, dtype=float)
    front2 = select_objective_columns(front3, OBJECTIVE_PAIR)
    front2 = front2[np.all(np.isfinite(front2), axis=1)]
    front2 = nondominated_front(front2)

    if len(front2) == 0:
        hv_val = 0.0
        igd_val = float('inf')
    else:
        hv_val = float(hv(front2))
        igd_val = float(igd(front2))

    return {
        'algo': algo_key,
        'seed': seed,
        'elapsed_s': elapsed,
        'best_fitness': float(best_fit),
        'front2': front2,
        'hv': hv_val,
        'igd': igd_val,
        'queries': int(fit.get_statistics()['api_queries']),
        'archive_size': int(len(front3)),
    }

ALGOS = ['NSGA2', 'NSGA2-PARTITIONED']
results = []
for algo in ALGOS:
    for r in range(NUM_RUNS):
        seed = SEED_BASE + r
        print(f'Running {algo} seed={seed} ...')
        res = run_one(algo, seed)
        results.append(res)
        print(
            f"  hv={res['hv']:.6g} igd={res['igd']:.6g} "
            f"archive={res['archive_size']} elapsed={res['elapsed_s']:.1f}s queries={res['queries']}"
        )

In [ ]:
# Summaries
rows = []
for r in results:
    rows.append({k: v for k, v in r.items() if k not in {'front2'}})
df = pd.DataFrame(rows)
df

In [ ]:
summary = df.groupby('algo').agg(
    hv_mean=('hv','mean'),
    hv_std=('hv','std'),
    igd_mean=('igd','mean'),
    igd_std=('igd','std'),
    elapsed_mean=('elapsed_s','mean'),
    queries_mean=('queries','mean'),
).reset_index()
summary

In [ ]:
# Boxplots for HV and IGD across runs
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x='algo', y='hv', ax=axes[0])
axes[0].set_title('Hypervolume (higher is better)')
axes[0].set_xlabel('')

sns.boxplot(data=df, x='algo', y='igd', ax=axes[1])
axes[1].set_title('IGD (lower is better)')
axes[1].set_xlabel('')
plt.tight_layout()
plt.show()

In [ ]:
# Choose the best run per algorithm (by HV) for plotting
best_by_algo = {}
for algo in ALGOS:
    sub = [r for r in results if r['algo'] == algo]
    best = max(sub, key=lambda x: x['hv'])
    best_by_algo[algo] = best
best_by_algo

In [ ]:
# Pareto front plot (2D)
x_name, y_name = OBJECTIVE_PAIR

def pretty_axis(name: str) -> str:
    if name == 'latency':
        return 'latency (normalized, lower better)'
    if name == 'energy':
        return 'energy (normalized, lower better)'
    if name == 'error':
        return 'error = 1 - acc/acc_baseline (lower better)'
    if name == 'accuracy':
        return 'accuracy (represented as error, lower better)'
    return f'{name} (lower better)'

def aggregate_front_for_algo(algo: str) -> np.ndarray:
    pts = [r['front2'] for r in results if r['algo'] == algo]
    if not pts:
        return np.empty((0, 2))
    union = np.vstack(pts)
    return nondominated_front(union)

plt.figure(figsize=(10, 7))
plt.scatter(ref_front_2d[:, 0], ref_front_2d[:, 1], s=18, c='gray', alpha=0.5, label='Reference (theoretical) Pareto')

colors = {'NSGA2': 'tab:red', 'NSGA2-PARTITIONED': 'tab:blue'}
for algo in ALGOS:
    # Best single run (by HV)
    best_front = best_by_algo[algo]['front2']
    plt.scatter(best_front[:, 0], best_front[:, 1], s=30, c=colors[algo], alpha=0.9, label=f'{algo} (best HV)')
    # Aggregated (union across runs, then non-dominated)
    agg_front = aggregate_front_for_algo(algo)
    if len(agg_front) > 0:
        plt.scatter(agg_front[:, 0], agg_front[:, 1], s=30, facecolors='none', edgecolors=colors[algo], linewidths=1.5, alpha=0.9, label=f'{algo} (union ND)')

plt.title(f'Pareto fronts in normalized objective space: {OBJECTIVE_PAIR}')
plt.xlabel(pretty_axis(x_name))
plt.ylabel(pretty_axis(y_name))
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Save outputs
out_dir = WORKDIR / 'nsga2_partitioned_comparison'
out_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(out_dir / 'per_run_metrics.csv', index=False)
summary.to_csv(out_dir / 'summary_metrics.csv', index=False)

with open(out_dir / 'settings.json', 'w') as f:
    json.dump({
        'REPO_URL': REPO_URL,
        'SEARCH_SPACE': SEARCH_SPACE,
        'DEVICE': DEVICE,
        'DATASET': DATASET,
        'NUM_RUNS': NUM_RUNS,
        'POPULATION_TOTAL': POPULATION_TOTAL,
        'ITERATIONS': ITERATIONS,
        'SEED_BASE': SEED_BASE,
        'OBJECTIVE_PAIR': OBJECTIVE_PAIR,
        'THEORETICAL_EVAL_LIMIT': THEORETICAL_EVAL_LIMIT,
        'ref_point': ref_point.tolist(),
        'reference_front_size': int(len(ref_front_2d)),
    }, f, indent=2)

print('Saved to:', out_dir)
list(out_dir.iterdir())